In [1]:
print('Loading Packages...\n')
%matplotlib osx
import sys, os          
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
import pandas
from collections import defaultdict
import scipy.stats
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'


df = pandas.read_csv('FullTankPMTGeometry.csv')
channel_number = []; PMT_type = []
for i in range(len(df['channel_num'])):
    channel_number.append(df['channel_num'][i])
    PMT_type.append(df['PMT_type'][i])
    

print('done')

Loading Packages...

done


### Michels, data

In [2]:
# import real data

filename = 'Michels_v1.root'

rootdata = uproot.open(filename)
TD = rootdata['Michels']
CPE_data = TD['cluster_pe'].array()
CCB_data = TD['cluster_Qb'].array()
CT_data = TD['cluster_time'].array()
CH_data = TD['cluster_hits'].array()
T_data = TD['hitT'].array()
PE_data = TD['hitPE'].array()
ID_data = TD['hitPMTID'].array()

print('Data loaded')

Data loaded


### AmBe, data

In [152]:
data_file = '../AmBe/AmBe_neutron_candidates/neutron_candidates_v5.root'

# ----------------------------------------------------- #

cluster_time = []; cluster_charge = []; cluster_QB = []; cluster_hits = []; 
hit_times = []; hit_charges = []; hit_ids = []
source_position = [[], [], []]      # x, y, z

# ----------------------------------------------------- #

# last index in each list are for the counts
poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
       [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
       [-75,100,0,0],[-75,50,0,0],[-75,0,0,0],[-75,-50,0,0],[-75,-100,0,0],   # Port 4
       [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]   # Port 3


print('\nLoading AmBe event data...')

neg_hits = []; neg_charge = []
with uproot.open(data_file) as file:

    Event = file["Neutrons"]
    CT = Event["cluster_time"].array()
    CPE = Event["cluster_charge"].array()
    CCB = Event["cluster_Qb"].array()
    CH = Event["cluster_hits"].array()
    hT = Event["hitT"].array()
    hPE = Event["hitPE"].array()
    hID = Event["hitID"].array()
    xp = Event["X_pos"].array()
    yp = Event["Y_pos"].array()
    zp = Event["Z_pos"].array()

    for i in trange(len(CT)):

        x1 = round(xp[i]); y1 = round(yp[i]); z1 = round(zp[i])
        
        # ENABLE for Type-only study
        #'''
        cc = 0; ch = 0
        for j in range(len(hPE[i])):
            indy = hID[i][j] - 332
            if PMT_type[indy] == 'Watchman':
                cc += hPE[i][j]
                ch += 1
        if ch > 0:  # enable to remove 0 bin
            cluster_charge.append(cc)   
            cluster_hits.append(ch)
            #'''
        
        # indent if above is uncommented ^ 
        cluster_time.append(CT[i])   
        #cluster_charge.append(CPE[i])   # comment if using Type-only
        cluster_QB.append(CCB[i])
        #cluster_hits.append(CH[i])      # comment if using Type-only
        hit_times.append(hT[i])
        hit_charges.append(hPE[i])
        hit_ids.append(hID[i])
        source_position[0].append(xp[i])
        source_position[1].append(yp[i])
        source_position[2].append(zp[i])

    
print('----------------------------------------------------------------\n')
print('We have: ', len(cluster_time), ' total AmBe neutron candidates\n')

print('\ndone')


Loading AmBe event data...


100%|███████████████████████████████████| 67940/67940 [01:03<00:00, 1064.34it/s]

----------------------------------------------------------------

We have:  67940  total AmBe neutron candidates


done


### Thru data

In [2]:
filename_thru = 'throughgoing_muons_v1.root'

rootdata_thru = uproot.open(filename_thru)
TD_thru = rootdata_thru['Muons']
CPE_thru_data = TD_thru['cluster_pe'].array()
CCB_thru_data = TD_thru['cluster_Qb'].array()
CT_thru_data = TD_thru['cluster_time'].array()
CH_thru_data = TD_thru['cluster_hits'].array()
Down_thru_data = TD_thru['downstream_pe'].array()
Up_thru_data = TD_thru['upsteam_pe'].array()
T_thru_data = TD_thru['hitT'].array()
PE_thru_data = TD_thru['hitPE'].array()
ID_thru_data = TD_thru['hitPMTID'].array()
ID_number_thru_data = TD_thru['hitID'].array()

print('Data loaded')

Data loaded


### Michels, MC

In [35]:
# extract dirt muons first

# directories to be used
MC_directories = [
    'simulated_data/QE_1.50_HM_WM_1.25/'#,
]#'simulated_data/QE_1.50_HM_1.50_2/'

# assemble full paths
file_paths = []
for MC_dir in MC_directories:
    full_paths = [os.path.join(MC_dir, fname) for fname in os.listdir(MC_dir)
                  if os.path.isfile(os.path.join(MC_dir, fname))]
    file_paths.extend(full_paths)
    

charge_balance = []; total_charge = []
muon_charge = []; muon_charge_balance = []; total_hits = []; total_time = []; avg_hit = []
dirt_muon_count = 0


count_cuts = [0, 0, 0, 0, 0, 0, 0, 0, 0]


# Process each file
for count, filepath in enumerate(file_paths):
    if count % 100 == 0:
        print(round(100 * count / len(file_paths), 2), '%')

    root = uproot.open(filepath)
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CT = T['clusterTime'].array()
    CN = T['clusterNumber'].array()
    CH = T['clusterHits'].array()
    X1 = T['hitX'].array()
    Y1 = T['hitY'].array()
    Z1 = T['hitZ'].array()
    T1 = T['hitT'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    TMC = T['TankMRDCoinc'].array()
    NV1 = T['NoVeto'].array()

    # need the TriggerTree for MRD activity
    Trig = root['phaseIITriggerTree']
    EN2 = Trig['eventNumber'].array()
    MRD2 = Trig['MRDhitT'].array()

    # build eventNumber -> MRDhitT map
    event_to_MRD = dict(zip(EN2, MRD2))

    # for the PMTWaveformSim tool, we skip blank events (no MCHits)
    # but we still have a mismatch between events with < 5 clustered hits and clusters
    
    count_cuts[0] += len(EN2)

    # loop over clusters
    for i in range(len(CEN)):    # enable trange for a single file

        #if i > 25000:
        #    break

        evt = CEN[i]

        # first, select dirt muon candidates

        # 1. No MRD activity
        mrd_hits = event_to_MRD.get(evt, [])  # empty list if event is missing (shouldn’t happen)
        if len(mrd_hits) != 0:
            continue

        # 2. Require Veto activity
        if NV1[i] == 1:
            continue

        # 3. No TankMRDCoinc (this should already be covered)
        if TMC[i] == 1:
            print('Seems like one got through to the MRD somehow!!!')
            continue
            
        count_cuts[1] += 1

        # 4. 1000 < cluster PE < 4000 (for now, lets disable this and investigate the charge distribution)
        if CPE[i] > 4000 or CPE[i] < 1000:
            continue
            
        count_cuts[2] += 1

        # 5. Prompt cluster, and can't be the only cluster (we need Michel candidates afterall!)
        if CN[i] != 0:
            if (i+1) == len(CN) or CN[i+1] == 0:     # if its the last cluster there aren't any more clusters to look for Michels
                continue

        # 6. (just for the sim) make sure the cluster time is under 150ns so its a prompt event
        if CT[i] > 150:
            continue

        # 7. Its the brightest event
        brightest = CPE[i]; flag = False
        for j in range(1, len(CEN) - i):
            if CN[j] == 0:
                break
            if CPE[j] > brightest:
                flag = True
                break
        if flag == True:
            continue
            
        count_cuts[3] += 1

        # 6. charge barycenter downstream
        a = 0
        for j in range(len(Z1[i])):
            a += Z1[i][j]*PE1[i][j]
        if a < 0:
            continue
            
        count_cuts[4] += 1

        # 7. Must have 100 PMT hits
        if CH[i] < 100:
            continue
            
        count_cuts[5] += 1

        # Any CCB cuts...
        if CCB[i] > 0.2 or CCB[i] < 0:
            continue
            
        count_cuts[6] += 1

        dirt_muon_count += 1
        muon_charge.append(CPE[i])
        muon_charge_balance.append(CCB[i])


        # Now, for the Michels!

        for j in range(i+1, len(CEN)):
            if CN[j] == 0:   # new event
                break
            time_diff = CT[j] - CT[i]
            if 200 < time_diff < 5000:
                count_cuts[7] += 1
                if CCB[j] < 0.18:#0.2:
                    count_cuts[8] += 1
                    '''    # uncomment for specific PMT-type analysis
                    a = 0; b = 0
                    for k in range(len(ID1[j])):
                        if PMT_type[ID1[j][k] - 332] == 'LUX' or PMT_type[ID1[j][k] - 332] == 'ETEL':
                            a += PE1[j][k]; b += 1      
                    total_charge.append(a)
                    total_hits.append(b)
                    #'''

                    avg_hit.append(np.average(PE1[j]))
                    charge_balance.append(CCB[j])
                    total_charge.append(CPE[j])        # comment for individual PMT type
                    total_hits.append(CH[j])           # and here
                    total_time.append(CT[j])
    

print('\n', dirt_muon_count, len(total_charge))
print('\ndone')

0.0 %
12.8 %
25.61 %
38.41 %
51.22 %
64.02 %
76.82 %
89.63 %

 11180 1601

done


In [36]:
print(count_cuts)
print(len(file_paths))

[85670, 67715, 21902, 21882, 14132, 13644, 11180, 5967, 1601]
781


### AmBe, MC

In [153]:
MC_directory = '../AmBe/AmBe_MC/pmt_tilting_v1/corrected_waveforms/QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25_corrected_waveforms/'
file_names = [file for file in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, file))]

QE_tag = '_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25'

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
          [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
          [-75,100,0,0],[-75,50,0,0],[-75,0,0,0],[-75,-50,0,0],[-75,-100,0,0],   # Port 4
          [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]   # Port 3


class Cluster_MC:
    def __init__(self,pe,Qb,nhits,time,hitPE,hitID,hitT,vtx,vty,vtz):
        self.pe = pe
        self.Qb = Qb
        self.nhits = nhits
        self.time = time
        self.hitPE = hitPE
        self.hitID = hitID
        self.hitT = hitT
        self.vtx = vtx
        self.vty = vty
        self.vtz = vtz
        
clusters_MC = []

count = 0; 
for file in trange(len(file_names)):

    root = uproot.open(MC_directory + file_names[file])
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CH = T['clusterHits'].array()
    CT = T['clusterTime'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    HT1 = T['hitT'].array()
    CN = T['clusterNumber'].array()
    
    # Port 5
    if file_names[file] == 'MC_AmBe_port5_z100' + QE_tag + '.root':
        xp = 0; yp = 100; zp = 0
    elif file_names[file] == 'MC_AmBe_port5_z50' + QE_tag + '.root':
        xp = 0; yp = 50; zp = 0
    elif file_names[file] == 'MC_AmBe_port5_z0' + QE_tag + '.root':
        xp = 0; yp = 0; zp = 0
    elif file_names[file] == 'MC_AmBe_port5_zminus50' + QE_tag + '.root':
        xp = 0; yp = -50; zp = 0
    elif file_names[file] == 'MC_AmBe_port5_zminus100' + QE_tag + '.root':
        xp = 0; yp = -100; zp = 0
        
    # Port 1
    elif file_names[file] == 'MC_AmBe_port1_z100' + QE_tag + '.root':
        xp = 0; yp = 100; zp = -75
    elif file_names[file] == 'MC_AmBe_port1_z50' + QE_tag + '.root':
        xp = 0; yp = 50; zp = -75
    elif file_names[file] == 'MC_AmBe_port1_z0' + QE_tag + '.root':
        xp = 0; yp = 0; zp = -75
    elif file_names[file] == 'MC_AmBe_port1_zminus50' + QE_tag + '.root':
        xp = 0; yp = -50; zp = -75
    elif file_names[file] == 'MC_AmBe_port1_zminus100' + QE_tag + '.root':
        xp = 0; yp = -100; zp = -75
        
    # Port 4
    elif file_names[file] == 'MC_AmBe_port4_z100' + QE_tag + '.root':
        xp = -75; yp = 100; zp = 0
    elif file_names[file] == 'MC_AmBe_port4_z50' + QE_tag + '.root':
        xp = -75; yp = 50; zp = 0
    elif file_names[file] == 'MC_AmBe_port4_z0' + QE_tag + '.root':
        xp = -75; yp = 0; zp = 0
    elif file_names[file] == 'MC_AmBe_port4_zminus50' + QE_tag + '.root':
        xp = -75; yp = -50; zp = 0
    elif file_names[file] == 'MC_AmBe_port4_zminus100' + QE_tag + '.root':
        xp = -75; yp = -100; zp = 0
        
    # Port 3
    elif file_names[file] == 'MC_AmBe_port3_z100' + QE_tag + '.root':
        xp = 0; yp = 100; zp = 102
    elif file_names[file] == 'MC_AmBe_port3_z50' + QE_tag + '.root':
        xp = 0; yp = 50; zp = 102
    elif file_names[file] == 'MC_AmBe_port3_z0' + QE_tag + '.root':
        xp = 0; yp = 0; zp = 102
    elif file_names[file] == 'MC_AmBe_port3_zminus50' + QE_tag + '.root':
        xp = 0; yp = -50; zp = 102
    elif file_names[file] == 'MC_AmBe_port3_zminus100' + QE_tag + '.root':
        xp = 0; yp = -100; zp = 102
        
        
    # in case you fuck up the file name, we don't want to populate xp, yp, zp with the arrays from the AmBe data cell
    else:
        print('INCORRECT FILE NAME!!!')
    

    for i in range(len(CPE)):
        
        # selection cuts
        if CPE[i] < 70 and 67000 > CT[i] > 2000 and CCB[i] < 0.4:
            
            # only cluster in the event
            if CN[i] != 0:
                continue
            else:    # zeroth cluster, make sure its the only cluster
                if i + 1 < len(CN):
                    if CN[i + 1] != 0:
                        continue
                        
            #'''  
            cc = 0; ch = 0
            for j in range(len(PE1[i])):
                indy = ID1[i][j] - 332
                if PMT_type[indy] == 'Watchman':# and ID1[i][j] in not_as_tilted:
                    cc += PE1[i][j]
                    ch += 1
            if ch > 0:       # for zero bin
                #'''

                # indent if applying individual PMT mask ^
                event = Cluster_MC(pe=cc,nhits=ch,
                                   #pe=CPE[i],nhits=CH[i],   # comment for Type-only
                                   Qb=CCB[i],time=CT[i],hitPE=PE1[i],hitID=ID1[i],hitT=HT1[i],
                                   vtx=xp,vty=yp,vtz=zp)
                                   #vtx=vx[CEN[i]],vty=vy[CEN[i]],vtz=vz[CEN[i]])
                clusters_MC.append(event)

                count += 1


print(count, 'events\n')
print('\ndone')

100%|███████████████████████████████████████████| 20/20 [04:07<00:00, 12.36s/it]

134868 events


done


### Thru MC

In [33]:
# for a single file 
#file = 'simulated_data/MC_dirt_muons_QE_1.25_0_500k.ntuple.root'

# for a group of files
#MC_directory = 'simulated_data/thru/QE_1.5_HM_1.25/'
MC_directory = 'simulated_data/thru/individual_tilted_v1/'
file_names = [file for file in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, file))]

charge_balance_thru = []; total_charge_thru = []; total_hits_thru = []
downstream_charge = []; upstream_charge = []; avg_hit_thru = []
thru_muon_count = 0

Top_charge = []; Bottom_charge = []

count = 0
for file in file_names:         # disable for a single file

    if count % 5 == 0:
        print(round(100*(count/len(file_names)),2), '%')   
    count += 1
        
    # indent for collection of files
    root = uproot.open(MC_directory + file)       # change to just 'file' for a single file
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CT = T['clusterTime'].array()
    CN = T['clusterNumber'].array()
    CH = T['clusterHits'].array()
    X1 = T['hitX'].array()
    Y1 = T['hitY'].array()
    Z1 = T['hitZ'].array()
    T1 = T['hitT'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    TMC = T['TankMRDCoinc'].array()
    NV1 = T['NoVeto'].array()

    # need the TriggerTree for MRD activity
    Trig = root['phaseIITriggerTree']
    EN2 = Trig['eventNumber'].array()
    MRD2 = Trig['numMRDTracks'].array()
    MRD_thru = Trig['MRDThrough'].array()

    # build eventNumber -> MRD map
    event_to_MRD = dict(zip(EN2, MRD2))
    event_to_MRD_thru = dict(zip(EN2, MRD_thru))

    # for the PMTWaveformSim tool, we skip blank events (no MCHits)
    # but we still have a mismatch between events with < 5 clustered hits and clusters

    # loop over clusters
    for i in range(len(CEN)):    # enable trange for a single file

        evt = CEN[i]

        
        # select through-going muon candidates

        
        # 1. MRD activity (coincidence, track, and through-going)
        mrd_track = event_to_MRD.get(evt, [])  # empty list if event is missing (shouldn’t happen)
        mrd_thru = event_to_MRD_thru.get(evt, [])   # this could be [] (blank), [True], or [False]
        
        if TMC[i] == 0:   # must have Tank/MRD coincidence
            continue
        if mrd_track != 1:   # not a single track
            continue
        if len(mrd_thru) == 0:  # not a through-going track
            continue
        if mrd_thru[0] != True:
            continue
        
        
        # 2. Require Veto activity
        if NV1[i] == 1:
            continue
        

        # 4. Prompt cluster (and occurs within the first 150ns)
        if CN[i] != 0:
            continue
        if CT[i] > 150:
            continue
        
        
        # 4. Its the brightest cluster in an event
        brightest = CPE[i]; flag = False
        for j in range(1, len(CEN) - i):
            if CN[j] == 0:
                break
            if CPE[j] > brightest:
                flag = True
                break
        if flag == True:
            continue
            
            
        # 5. Charge balance < 0.2
        if CCB[i] > 0.2 or CCB[i] < 0:
            continue
            
            
        # 6. Must have 100 PMT hits
        if CH[i] < 100:
            continue
            
            
        # 7. charge barycenter downstream
        a = 0; down_c = 0; up_c = 0
        for j in range(len(Z1[i])):
            a += Z1[i][j]*PE1[i][j]
            if Z1[i][j] > 0:
                down_c += PE1[i][j]
            else:
                up_c += PE1[i][j]
        if a < 0:
            continue
        #'''    
        # uncomment for specific PMT-type analysis
        e = 0; f = 0
        gg = 0; zz = 0
        for k in range(len(ID1[i])):
            if PMT_type[ID1[i][k] - 332] == 'Hamamatsu':
                e += PE1[i][k]; f += 1     
                if Y1[i][k] > 0 and Z1[i][k] > 0:
                    gg += PE1[i][k]   # Top and downstream
                elif Y1[i][k] < 0 and Z1[i][k] > 0:
                    zz += PE1[i][k]   # Bottom and downstream
                
        if f > 0:   # indent below if uncommenting
            #''' 
            total_charge_thru.append(e)
            total_hits_thru.append(f)

            Top_charge.append(gg); Bottom_charge.append(zz)


            thru_muon_count += 1

            avg_hit_thru.append(np.average(PE1[i]))
            charge_balance_thru.append(CCB[i])
            #total_charge_thru.append(CPE[i])        # comment for PMT-type
            #total_hits_thru.append(CH[i])           # comment here too

            downstream_charge.append(down_c)
            upstream_charge.append(up_c)
    

print('\n', thru_muon_count)
print('\ndone')

0.0 %
3.25 %
6.49 %
9.74 %
12.99 %
16.23 %
19.48 %
22.73 %
25.97 %
29.22 %
32.47 %
35.71 %
38.96 %
42.21 %
45.45 %
48.7 %
51.95 %
55.19 %
58.44 %
61.69 %
64.94 %
68.18 %
71.43 %
74.68 %
77.92 %
81.17 %
84.42 %
87.66 %
90.91 %
94.16 %
97.4 %

 6379

done


In [19]:
fig, ax = plt.subplots(figsize=(5, 4))
#plt.hist(Top_charge, bins = 100, histtype = 'step', label = 'MC (top)', linewidth = 2)
#plt.hist(Bottom_charge, bins = 100, histtype = 'step', label = 'MC (bottom)', linewidth = 2)
plt.hist(CPE_thru_data, bins = 100, histtype = 'stepfilled', color = 'k', alpha = 0.3, linewidth = 2, label = 'Data')
plt.hist(total_charge_thru, bins = 100, histtype = 'step', color = 'darkred', linewidth = 2, label = 'MC')
plt.legend(fontsize = 12, frameon = False)
plt.xlim([0,8000])
plt.xlabel('cluster charge [p.e.]')
plt.tight_layout()
#plt.savefig('plots/comparison/individual_tilt/thru cluster charge MC data.png',
#            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

### Comparison Fits

In [34]:
# QE and CCB adjustments

dummy = []
for i in range(len(total_charge_thru)):
    #if total_charge[i] > 0:
        #if charge_balance[i] < 0.2:
    dummy.append(total_charge_thru[i])

#'''
dummy_data = []
for i in trange(len(CPE_thru_data)):
    #if CCB_data[i] < 0.2:
    c = 0
    for j in range(len(ID_thru_data[i])):
        if ID_thru_data[i][j] == 3:
            c += PE_thru_data[i][j]
    if c > 0:
        dummy_data.append(c)
        #dummy_data.append(CPE_data[i])
#'''

print(len(dummy_data), len(dummy))
print(min(dummy_data), max(dummy_data))
print(min(dummy), max(dummy))

plt.hist(dummy, bins = 50, histtype = 'step', density = True, label = 'MC', linewidth = 2)
plt.hist(dummy_data, bins = 50, histtype = 'step', density = True, label = 'Data', linewidth = 2)
plt.legend(fontsize = 12, frameon = False)
plt.xlabel('cluste charge [p.e]')
plt.show()

#plt.hist(CPE_data, bins = 50, histtype = 'step', density = True, label = 'Data')
#plt.hist(total_charge, bins = 50, histtype = 'step', density = True, label = 'MC')
#plt.legend(fontsize = 12)
#plt.show()

100%|██████████████████████████████████████| 6154/6154 [00:43<00:00, 141.24it/s]

6154 6379
84.03487176647621 6781.560301543726
116.8288058745624 5895.815345049228


In [28]:
# AmBe adjustment

dummy= []
for i in range(len(total_charge_thru)):
    dummy.append(total_charge_thru[i] * 0.7)

plt.hist(dummy, bins = 20, histtype = 'step', density = True, label = 'MC')
plt.hist(CPE_thru_data, bins = 20, histtype = 'step', density = True, label = 'data')
plt.legend(fontsize = 12)
plt.show()

#plt.hist(CPE_data, bins = 50, histtype = 'step', density = True, label = 'Data')
#plt.hist(total_charge, bins = 50, histtype = 'step', density = True, label = 'MC')
#plt.legend(fontsize = 12)
#plt.show()

In [30]:
# this snippet will produce a cumulative PMT charge plot for both Michels + AmBe neutrons, and provide chi-sq agreements

import random

plot_save_dir = 'plots/comparison/individual_tilt/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False    # enable for verbosity

which_one = 'Thru'

# --- AmBe ---- #
plot_color_Ambe = 'tab:purple'
mode = 'charge'         # 'charge' or 'hits'
# # # # # # # # # # # # # # # # # # # #
QE_factor_AmBe = 1.0    # default = 1.0
# # # # # # # # # # # # # # # # # # # #
bin_size_AmBe = 0.25     # default: 1.0
binning_AmBe = np.arange(0, 8.25, bin_size_AmBe)    # default: 5, 65 or 70
xlim_low_AmBe = 0; xlim_high_AmBe = 9    # plot xlim window: default for all: [0,75]

# --- Michels ---- #
plot_color_Michel = 'tab:orange'
bin_size_Michel = 25     # default 20
binning_Michel = np.arange(40, 615, bin_size_Michel)    # default: 60, 500
xlim_low_Michel = 0; xlim_high_Michel = 700    # plot xlim window: default for all: [20,650]

# --- Thru ---- #
plot_color_thru = 'tab:red'
bin_size_thru = 100     # default 100
binning_thru = np.arange(500, 6500, bin_size_thru)    # default: 500, 6500
xlim_low_thru = 0; xlim_high_thru = 8000    # plot xlim window: default for all: [0,8000]

# # # # # # # # # # # # # # # # # # # # # # # # # #

# useful functions

def get_common_positions(d1, d2):
    common_keys = set(d1.keys()) & set(d2.keys())
    filtered = []
    for k in common_keys:
        if len(d1[k]) > 0 and len(d2[k]) > 0:
            filtered.append(k)
    return filtered

def balance_dataset_with_keys(pos_dict, keys, n_per_pos):
    all_entries = []
    for k in sorted(keys):
        if len(pos_dict[k]) >= n_per_pos:
            all_entries.extend(random.sample(pos_dict[k], n_per_pos))
    return all_entries

def generate_merged_bins(bins, counts_data, counts_mc, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc = counts_mc[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc < min_counts) and j < len(counts_data):
            sum_d += counts_data[j]
            sum_mc += counts_mc[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


def balance_samples(list1, list2):
    """
    Randomly samples both lists to ensure the same number of entries.
    Returns the balanced versions.
    """
    list1 = list(list1)
    list2 = list(list2)
    n = min(len(list1), len(list2))
    return random.sample(list1, n), random.sample(list2, n)
    
    
# # # # # # # # # # # # # # # # # # # # # # # # # #

if which_one == 'AmBe':

    # AmBe plot
    # # # # # # # # # # # # # # # # # # # # # # # #
    print('\n------- AmBe plot -------\n')
    data_dict = defaultdict(list)
    mc_dict = defaultdict(list)

    # Prepare full sample
    data_all = []; mc_all = []; mc_control_all = []

    # we must ensure equal counts at each position source for both MC and data

    # Key: (x, y, z)
    for i in range(len(cluster_charge)):
        key = (source_position[0][i], source_position[1][i], source_position[2][i])
        if mode == 'charge':
            data_dict[key].append(cluster_charge[i])
        elif mode == 'hits':
            data_dict[key].append(cluster_hits[i])

    # adjust the tuning here (mc.pe)
    for mc in clusters_MC:
        key = (mc.vtx, mc.vty, mc.vtz)
        if mode == 'charge':
            mc_dict[key].append(mc.pe * QE_factor_AmBe)
        elif mode == 'hits':
            mc_dict[key].append(mc.nhits)

    mc_keys = list(mc_dict.keys())
    data_keys = list(data_dict.keys())

    # Find shared, valid keys
    valid_keys = get_common_positions(data_dict, mc_dict)

    # Get minimum across both datasets for just those positions
    min_data = min(len(data_dict[k]) for k in valid_keys)
    min_mc   = min(len(mc_dict[k]) for k in valid_keys)
    min_common = min(min_data, min_mc)

    # Now sample equally per position from these valid keys
    data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
    mc_all   = balance_dataset_with_keys(mc_dict, valid_keys, min_common)

    # Sanity check
    assert len(data_all) == len(mc_all), "Data and MC not balanced after enforced key matching!"

    if verbose:
        print('\nCounts after sampling:', len(data_all), len(mc_all), '\n')


    # MC counts and bins
    counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
    raw_counts_mc, bins_mc = np.histogram(mc_all, bins=binning_AmBe)
    counts_mc = raw_counts_mc

    fill_bins = generate_merged_bins(binning_AmBe, counts_data, counts_mc, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
    raw_counts_mc, bins_mc = np.histogram(mc_all, bins=fill_bins)
    counts_mc = raw_counts_mc

    if verbose:
        print("\nMerged bin edges:")
        for i in range(len(fill_bins) - 1):
            print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")

    # Errors
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
    data_errors = np.sqrt(counts_data)
    mc_errors = np.sqrt(raw_counts_mc)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


    # Compute chi-squared
    chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nChi2/NDF: {chi2 / ndf:.2f}")


    # calculate residuals and errors
    mc_data = []; mc_errors_plot = []; x_info = []

    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue  # skip bin with no data
        x_info.append(binscenters[i])

        # MC
        if counts_mc[i] == 0:
            mc_data.append(0)
            mc_errors_plot.append(0)
        else:
            ratio_mc = counts_mc[i] / counts_data[i]
            error_mc = ratio_mc * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors[i] / counts_mc[i])**2
            )
            mc_data.append(ratio_mc)
            mc_errors_plot.append(error_mc)


    fig = plt.figure(figsize=(6, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    ax1.set_title('MC / Data AmBe neutrons', fontsize = 16)
    ax2.set_xlabel('Cluster charge [p.e.]', fontsize = 14)    # FIX
    ax1.set_ylabel(f'clusters / {bin_size_AmBe} p.e. bin', fontsize = 14)
    ax2.set_ylabel('MC / Data', fontsize = 14)

    ax1.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_ylim([0, 2]); 

    # MC histogram
    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc, histtype='step', color=plot_color_Ambe, label='MC', linewidth=2)
    ax1.fill_between(      # shaded MC error band (turn off if needed)
        binscenters,
        counts_mc - mc_errors,
        counts_mc + mc_errors,
        step='mid',
        color=plot_color_Ambe,
        alpha=0.3
    )

    # data histogram
    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

    # subplot below for residuals
    ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize = 1,
                 fmt='o', color=plot_color_Ambe)
    ax2.axhline(1, linestyle = 'dashed', color = 'k')

    ax1.text(0.66, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color=plot_color_Ambe)
    ax1.text(0.66, 0.45, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
    ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')

    plt.tight_layout()
    path_str = plot_save_dir + 'MC_Data_AmBe_neutrons_WATCHMAN_cluster_pe_' + plot_save_name_base + '.png'
    #plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()

# # # # # # # # # # # # # # # # # # # # # # # #
    
elif which_one == 'Michel':
    
    # total_charge (MC)
    # CPE_data (data)
        
    # Michel plot
    # # # # # # # # # # # # # # # # # # # # # # # #
    print('\n------- Michel plot -------\n')
    
    #balanced_MC, balanced_data = balance_samples(total_charge, CPE_data)
    balanced_MC, balanced_data = balance_samples(dummy, dummy_data)
    
     # Sanity check
    assert len(balanced_MC) == len(balanced_data), "Sampling failed to balance datasets!"

    if verbose:
        print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


    # MC counts and bins
    counts_data, bins_data = np.histogram(balanced_data, bins=binning_Michel)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_Michel)
    counts_mc = raw_counts_mc

    fill_bins = generate_merged_bins(binning_Michel, counts_data, counts_mc, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
    counts_mc = raw_counts_mc

    if verbose:
        print("\nMerged bin edges:")
        for i in range(len(fill_bins) - 1):
            print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")

    # Errors
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
    data_errors = np.sqrt(counts_data)
    mc_errors = np.sqrt(raw_counts_mc)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


    # Compute chi-squared
    chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nChi2/NDF: {chi2 / ndf:.2f}")


    # calculate residuals and errors
    mc_data = []; mc_errors_plot = []; x_info = []

    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue  # skip bin with no data
        x_info.append(binscenters[i])

        # MC
        if counts_mc[i] == 0:
            mc_data.append(0)
            mc_errors_plot.append(0)
        else:
            ratio_mc = counts_mc[i] / counts_data[i]
            error_mc = ratio_mc * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors[i] / counts_mc[i])**2
            )
            mc_data.append(ratio_mc)
            mc_errors_plot.append(error_mc)


    fig = plt.figure(figsize=(6, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    ax1.set_title('MC / Data Michels', fontsize = 16)
    ax2.set_xlabel('Cluster charge [p.e.]', fontsize = 14)    # FIX
    ax1.set_ylabel(f'clusters / {bin_size_Michel} p.e. bin', fontsize = 14)
    ax2.set_ylabel('MC / Data', fontsize = 14)

    ax1.set_xlim([xlim_low_Michel, xlim_high_Michel]); ax2.set_xlim([xlim_low_Michel, xlim_high_Michel]); ax2.set_ylim([0, 2]); 

    # MC histogram
    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc, histtype='step', color=plot_color_Michel, label='MC', linewidth=2)
    
    #ax1.fill_between(      # shaded MC error band (turn off if needed)    (OLD WAY)
    #    binscenters,
    #    counts_mc - mc_errors,
    #    counts_mc + mc_errors,
    #    step='mid',
    #    color=plot_color_Michel,
    #    alpha=0.3
    #)
    
    # Pad arrays to match the length of fill_bins (N+1) to ensure all bins (after re-binning) have error bands
    counts_mc_padded = np.append(counts_mc, counts_mc[-1])
    mc_errors_padded = np.append(mc_errors, mc_errors[-1])

    # Now create upper and lower edges
    mc_low  = counts_mc_padded - mc_errors_padded
    mc_high = counts_mc_padded + mc_errors_padded

    # Plot using step='post' to include final bin
    ax1.fill_between(
        fill_bins,
        mc_low,
        mc_high,
        step='post',
        color=plot_color_Michel,
        alpha=0.3
    )

    # data histogram
    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

    # subplot below for residuals
    ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize = 1,
                 fmt='o', color=plot_color_Michel)
    ax2.axhline(1, linestyle = 'dashed', color = 'k')

    ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color=plot_color_Michel)
    ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
    ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')

    plt.tight_layout()
    path_str = plot_save_dir + 'CCB_0.2/more_stats/QE_0.95_MC_Data_Michels_total_cluster_pe_' + plot_save_name_base + '_CCB_0.18_MC.png'
    #path_str = plot_save_dir + 'CCB_0.18/MC_Data_Michels_total_cluster_hits_' + plot_save_name_base + '_CCB_0.18_MC.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
    
    
# # # # # # # # # # # # # # # # # # # # # # # #
    
elif which_one == 'Thru':
    
    # CPE_thru_data
    # total_charge_thru
        
    # Through-going plot
    # # # # # # # # # # # # # # # # # # # # # # # #
    print('\n------- Throughgoing plot -------\n')
    
    balanced_MC, balanced_data = balance_samples(total_charge_thru, CPE_thru_data)
    #balanced_MC, balanced_data = balance_samples(dummy, CPE_thru_data)
    
     # Sanity check
    assert len(balanced_MC) == len(balanced_data), "Sampling failed to balance datasets!"

    if verbose:
        print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


    # MC counts and bins
    counts_data, bins_data = np.histogram(balanced_data, bins=binning_thru)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_thru)
    counts_mc = raw_counts_mc

    fill_bins = generate_merged_bins(binning_thru, counts_data, counts_mc, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
    counts_mc = raw_counts_mc

    if verbose:
        print("\nMerged bin edges:")
        for i in range(len(fill_bins) - 1):
            print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")

    # Errors
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
    data_errors = np.sqrt(counts_data)
    mc_errors = np.sqrt(raw_counts_mc)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


    # Compute chi-squared
    chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nChi2/NDF: {chi2 / ndf:.2f}")


    # calculate residuals and errors
    mc_data = []; mc_errors_plot = []; x_info = []

    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue  # skip bin with no data
        x_info.append(binscenters[i])

        # MC
        if counts_mc[i] == 0:
            mc_data.append(0)
            mc_errors_plot.append(0)
        else:
            ratio_mc = counts_mc[i] / counts_data[i]
            error_mc = ratio_mc * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors[i] / counts_mc[i])**2
            )
            mc_data.append(ratio_mc)
            mc_errors_plot.append(error_mc)


    fig = plt.figure(figsize=(6, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    ax1.set_title('MC / Data Throughgoing Muons', fontsize = 16)
    ax2.set_xlabel('Cluster charge [p.e.]', fontsize = 14)    # FIX
    ax1.set_ylabel(f'clusters / {bin_size_thru} p.e. bin', fontsize = 14)
    ax2.set_ylabel('MC / Data', fontsize = 14)

    ax1.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_ylim([0, 2]); 

    # MC histogram
    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc, histtype='step', color=plot_color_thru, label='MC', linewidth=2)
    
    #ax1.fill_between(      # shaded MC error band (turn off if needed)    (OLD WAY)
    #    binscenters,
    #    counts_mc - mc_errors,
    #    counts_mc + mc_errors,
    #    step='mid',
    #    color=plot_color_Michel,
    #    alpha=0.3
    #)
    
    # Pad arrays to match the length of fill_bins (N+1) to ensure all bins (after re-binning) have error bands
    counts_mc_padded = np.append(counts_mc, counts_mc[-1])
    mc_errors_padded = np.append(mc_errors, mc_errors[-1])

    # Now create upper and lower edges
    mc_low  = counts_mc_padded - mc_errors_padded
    mc_high = counts_mc_padded + mc_errors_padded

    # Plot using step='post' to include final bin
    ax1.fill_between(
        fill_bins,
        mc_low,
        mc_high,
        step='post',
        color=plot_color_thru,
        alpha=0.3
    )

    # data histogram
    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

    # subplot below for residuals
    ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize = 1,
                 fmt='o', color=plot_color_thru)
    ax2.axhline(1, linestyle = 'dashed', color = 'k')

    ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color=plot_color_thru)
    ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
    ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')

    plt.tight_layout()
    path_str = plot_save_dir + 'MC_Data_Thru_cluster_pe_' + plot_save_name_base + '.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
        
        


print('\ndone')


------- Throughgoing plot -------


Chi2/NDF: 61.39

done


In [47]:
# QE adjustments

df = pandas.read_csv('FullTankPMTGeometry.csv')
y_pos = []; z_pos = []
for i in range(len(df['channel_num'])):
    y_pos.append(df['y_pos'][i] + 0.1446)
    z_pos.append(df['z_pos'][i] - 1.681)


dummy = []
for i in range(len(Bottom_charge)):
    if Bottom_charge[i] > 0:
        dummy.append(Bottom_charge[i])

#'''
dummy_data = []
for i in trange(len(CPE_thru_data)):
    c = 0
    for j in range(len(ID_thru_data[i])):
        if ID_thru_data[i][j] == 3:   # Watchboy
            if y_pos[int(ID_number_thru_data[i][j])-332] < 0 and z_pos[int(ID_number_thru_data[i][j])-332] > 0:
                c += PE_thru_data[i][j]
    if c > 0:
        dummy_data.append(c)
    #dummy_data.append(Up_thru_data[i] / Down_thru_data[i])
#'''

print(len(dummy_data), len(dummy))
print(min(dummy_data), max(dummy_data))
#print(min(dummy), max(dummy))

plt.hist(dummy, bins = 50, histtype = 'step', density = True, label = 'MC', linewidth = 2)
plt.hist(dummy_data, bins = 50, histtype = 'step', density = True, label = 'Data', linewidth = 2)
plt.legend(fontsize = 12, frameon = False)
plt.xlabel('charge [p.e.]')
plt.title('Hamamatsu, Bottom (Y < 0) PMTs only')
plt.savefig('plots/comparison/thru_investigation/Hamamatsu bottom charge thru muons.png',dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('\ndone')

100%|██████████████████████████████████████| 6154/6154 [00:51<00:00, 120.29it/s]


6154 10289
13.453552288546348 2980.488091506165

done


In [32]:
# upstream / downstream ratio + top / bottom investigation

import random

plot_save_dir = 'plots/comparison/individual_tilt/'
plot_save_name_base = 'individual_tilt_QE_1.50_HM_1.25'

verbose = False    # enable for verbosity

which_one = 'Thru'

# --- Thru ---- #
plot_color_thru = 'tab:red'
# upstream / downstream ratio
bin_size_thru = 0.05     # default 100
binning_thru = np.arange(0.1, 0.95, bin_size_thru)    # default: 500, 6500
xlim_low_thru = 0; xlim_high_thru = 1.00    # plot xlim window: default for all: [0,8000]

# # # # # # # # # # # # # # # # # # # # # # # # # #

# create the ratio
dummy = []; dummy_data = []
for i in range(len(upstream_charge)):
    dummy.append(upstream_charge[i] / downstream_charge[i])
for i in range(len(Up_thru_data)):
    dummy_data.append(Up_thru_data[i] / Down_thru_data[i])
    

# useful functions

def get_common_positions(d1, d2):
    common_keys = set(d1.keys()) & set(d2.keys())
    filtered = []
    for k in common_keys:
        if len(d1[k]) > 0 and len(d2[k]) > 0:
            filtered.append(k)
    return filtered

def balance_dataset_with_keys(pos_dict, keys, n_per_pos):
    all_entries = []
    for k in sorted(keys):
        if len(pos_dict[k]) >= n_per_pos:
            all_entries.extend(random.sample(pos_dict[k], n_per_pos))
    return all_entries

def generate_merged_bins(bins, counts_data, counts_mc, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc = counts_mc[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc < min_counts) and j < len(counts_data):
            sum_d += counts_data[j]
            sum_mc += counts_mc[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


def balance_samples(list1, list2):
    """
    Randomly samples both lists to ensure the same number of entries.
    Returns the balanced versions.
    """
    list1 = list(list1)
    list2 = list(list2)
    n = min(len(list1), len(list2))
    return random.sample(list1, n), random.sample(list2, n)
    
    
# # # # # # # # # # # # # # # # # # # # # # # #
    
if which_one == 'Thru':
    
    # CPE_thru_data
    # total_charge_thru
        
    # Through-going plot
    # # # # # # # # # # # # # # # # # # # # # # # #
    print('\n------- Throughgoing plot -------\n')
    
    #balanced_MC, balanced_data = balance_samples(downstream_charge, Down_thru_data)
    balanced_MC, balanced_data = balance_samples(dummy, dummy_data)
    
     # Sanity check
    assert len(balanced_MC) == len(balanced_data), "Sampling failed to balance datasets!"

    if verbose:
        print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


    # MC counts and bins
    counts_data, bins_data = np.histogram(balanced_data, bins=binning_thru)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_thru)
    counts_mc = raw_counts_mc

    fill_bins = generate_merged_bins(binning_thru, counts_data, counts_mc, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
    raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
    counts_mc = raw_counts_mc

    if verbose:
        print("\nMerged bin edges:")
        for i in range(len(fill_bins) - 1):
            print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")

    # Errors
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
    data_errors = np.sqrt(counts_data)
    mc_errors = np.sqrt(raw_counts_mc)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


    # Compute chi-squared
    chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nChi2/NDF: {chi2 / ndf:.2f}")


    # calculate residuals and errors
    mc_data = []; mc_errors_plot = []; x_info = []

    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue  # skip bin with no data
        x_info.append(binscenters[i])

        # MC
        if counts_mc[i] == 0:
            mc_data.append(0)
            mc_errors_plot.append(0)
        else:
            ratio_mc = counts_mc[i] / counts_data[i]
            error_mc = ratio_mc * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors[i] / counts_mc[i])**2
            )
            mc_data.append(ratio_mc)
            mc_errors_plot.append(error_mc)


    fig = plt.figure(figsize=(6, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    ax1.set_title('MC / Data Throughgoing Muons', fontsize = 16)
    ax2.set_xlabel('upstream / downstream charge ratio', fontsize = 14)    # FIX
    ax1.set_ylabel(f'clusters / {bin_size_thru} p.e. bin', fontsize = 14)
    ax2.set_ylabel('MC / Data', fontsize = 14)

    ax1.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_ylim([0, 2]); 

    # MC histogram
    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc, histtype='step', color=plot_color_thru, label='MC', linewidth=2)

    # Pad arrays to match the length of fill_bins (N+1) to ensure all bins (after re-binning) have error bands
    counts_mc_padded = np.append(counts_mc, counts_mc[-1])
    mc_errors_padded = np.append(mc_errors, mc_errors[-1])

    # Now create upper and lower edges
    mc_low  = counts_mc_padded - mc_errors_padded
    mc_high = counts_mc_padded + mc_errors_padded

    # Plot using step='post' to include final bin
    ax1.fill_between(
        fill_bins,
        mc_low,
        mc_high,
        step='post',
        color=plot_color_thru,
        alpha=0.3
    )

    # data histogram
    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

    # subplot below for residuals
    ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize = 1,
                 fmt='o', color=plot_color_thru)
    ax2.axhline(1, linestyle = 'dashed', color = 'k')

    ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color=plot_color_thru)
    ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
    ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')

    plt.tight_layout()
    path_str = plot_save_dir + 'upstream_MC_Data_Thru_upstream_downstream_ratio_' + plot_save_name_base + '.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
        


print('\ndone')


------- Throughgoing plot -------


Chi2/NDF: 13.59

done


In [ ]:
# chi_sq plot

QE_vals = [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15]
chisq_dof_AmBe = [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57]
chisq_dof_Michel = [124/11, 46/12, 22/12, 42/12, 85/12, 155/12, 208/12]

QE_vals_thru = [0.80, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1]
chisq_dof_thru = [11.05, 3.36, 2.69, 6.42, 16.06, 25.73, 36.07]

plt.figure(figsize=(4, 3))
plt.plot(QE_vals[1:], chisq_dof_AmBe[1:], marker='o', linestyle='-', color = 'tab:purple', label='AmBe')
plt.plot(QE_vals, chisq_dof_Michel, marker='o', linestyle='-', color = 'tab:orange', label='Michels')
plt.plot(QE_vals_thru, chisq_dof_thru, marker='o', linestyle='-', color = 'tab:red', label='Thru-' + r'$\mu$')

plt.xlabel(r'$r_{CE}$')
plt.ylabel(r'$\chi^{2} / dof$')
#plt.grid(True)
plt.legend(fontsize = 12, frameon = False)
#plt.gca().invert_xaxis()  # to align with above plots
#plt.xlim([0.5, 1.2])
plt.tight_layout()
path_str = 'plots/CHISQ _ AMBE and MICHEL AND THRU _ QE_1.50_HM_1.25.png'
plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

In [134]:
# chi_sq plot per PMT type

def plot_chi2(QE_vals, chisqdof, labels, colors, TYPE):
    
    plt.figure(figsize=(4, 3))
    
    for i in range(len(QE_vals)):
        plt.plot(QE_vals[i], chisqdof[i], marker='o', linestyle='-', color = colors[i], label = labels[i])

    plt.xlabel(r'$r_{CE}$')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize = 12, frameon = True, loc = 'upper right')
    #plt.xlim([0.6,1.3])
    #plt.ylim([1,10])
    plt.title(TYPE, fontsize = 12)
    plt.tight_layout()
    path_str = 'plots/CHISQ ' + TYPE + ' _ AMBE and MICHEL AND THRU _ QE_1.50_HM_1.25.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
    
    
labels = [
    'AmBe',
    'Michel',
    'Thru-' + r'$\mu$' 
]

colors = [
    'tab:purple',
    'tab:orange',
    'tab:red'
]


# Watchboy

QE_vals_WB = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],    # AmBe
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],      # Michels
    [0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]  # Thru
]

chisq_WB = [
    [26.57, 13.58, 9.08, 6.16, 7.65, 12.6, 19.35],  # AmBe
    [7.06, 4.06, 2.35, 2.22, 2.1, 4.04, 5.49],  # Michels
    [37.8, 16.83, 6.45, 6.68, 15.48, 26.05, 41.04, 54.86]  # Thru
]

#plot_chi2(QE_vals_WB, chisq_WB, labels, colors, 'WATCHBOY')


# Hamamatsu

QE_vals_HM = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],    # AmBe
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],      # Michels
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15]   # Thru
]

chisq_HM = [
    [27.83, 15.76, 9.01, 3.56, 1.51, 2.69, 7.18, 11.48],  # AmBe
    [8.79, 3.85, 2.33, 0.96, 1.12, 2.14, 2.76, 4.99],  # Michels
    [25.17, 12.09, 4.8, 2.53, 6.85, 14.58, 22.22]  # Thru
]

#plot_chi2(QE_vals_HM, chisq_HM, labels, colors, 'HAMAMATSU')



# LUX + ETEL

QE_vals_LE = [
    [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],    # AmBe
    [0.75, 0.80, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1],      # Michels
    [0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]   # Thru
]

chisq_LE = [
    [8.06, 5.01, 3.93, 1.48, 1.63, 2.61, 4.13, 5.81, 8.21],  # AmBe
    [4.83, 2.29, 1.52, 1.33, 2.2, 2.41, 4.45, 6.41],  # Michels
    [6.79, 3.66, 1.87, 2.31, 2.71, 4.44, 7.45]  # Thru
]

plot_chi2(QE_vals_LE, chisq_LE, labels, colors, 'LUX + ETEL')



print('\nPlots saved')


Plots saved


In [189]:
# AmBe chisq fit

def plot_chi2_AmBe(QE_vals, chisqdof, labels, linestyles, linewidths, markers, colors, TYPE):
    
    plt.figure(figsize=(5, 3))
    
    for i in range(len(QE_vals)):
        plt.plot(QE_vals[i], chisqdof[i], marker='o', linestyle=linestyles[i], color=colors[i], label=labels[i], linewidth=linewidths[i])

    plt.xlabel(r'$r_{CE}$')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    #plt.xlim([0.75,1.25])
    #plt.ylim([1,10])
    plt.title(TYPE, fontsize = 14)
    plt.tight_layout()
    path_str = 'plots/CHISQ ' + TYPE + ' _ AMBE _ QE_1.50_HM_1.25.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
    
    
labels = [
    'Total',
    'HM',
    'WB',
    'LUX + ETEL'
]

linestyles = [
    '-',
    'dashed',
    'dashed',
    'dashed'
]

linewidths = [
    2,
    1,
    1,
    1
]

markers = [
    'o',
    'd',
    's',
    'v'
]

colors = [
    'black',
    'tab:purple',
    'dodgerblue',
    'tab:red'
]


chisq_dof_AmBe = [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57]


QE_vals = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],  # total
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2], # HM
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],  # WB
    [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2]  # ETEL + LUX
    
]

chisq = [
    [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57],  # total
    [27.83, 15.76, 9.01, 3.56, 1.51, 2.69, 7.18, 11.48],  # HM
    [26.57, 13.58, 9.08, 6.16, 7.65, 12.6, 19.35],  # WB
    [8.06, 5.01, 3.93, 1.48, 1.63, 2.61, 4.13, 5.81, 8.21]
    
]

plot_chi2_AmBe(QE_vals, chisq, labels, linestyles, linewidths, markers, colors, 'AmBe neutrons')



print('\nPlots saved')


Plots saved


In [23]:
# by Port position and depth

# AmBe chisq fit

def plot_chi2_AmBe(chisqdof, labels, colors, averages):
    
    plt.figure(figsize=(5, 3))
    
    depths_cm = [+100, +50, 0, -50, -100]
    for i in range(len(chisqdof)):  # loop through Port positions
        if i == 3:   # Port 3 omitted
            plt.plot([+50, 0, -50, -100], chisqdof[i], marker='o', linestyle='-', color=colors[i], label=labels[i], linewidth=1)
        else:
            plt.plot(depths_cm, chisqdof[i], marker='o', linestyle='-', color=colors[i], label=labels[i], linewidth=1)
        
    plt.plot(depths_cm, averages, color = 'k', zorder = 20, marker = '+', label = 'Cumulative', linestyle = 'dashed')

    plt.xlabel('Source Depth [cm]')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.ylim([0,25])
    plt.title('AmBe neutrons (Default Tilt)', fontsize = 14)
    plt.gca().invert_xaxis()
    plt.tight_layout()
    path_str = 'plots/geometry_comparison/AmBe/CHISQ DEFAULT TILTING _ AMBE _ BY POSITION _ QE_1.50_HM_1.25.png'
    plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
    plt.show()
    
    
labels = [
    'Port 5',
    'Port 1',
    'Port 4',
    'Port 3'
]

colors = [
    'tab:orange',
    'dodgerblue',
    'tab:purple',
    'tab:red'
]

''' # approx tilted
chisq_dof_AmBe = [[12/14, 76/14, 16/13, 33/14, 11/13],    # Port 5
                  [53/14, 51/14, 33/14, 46/14, 74/13],    # Port 1
                  [20/14, 16/14, 22/14, 24/14, 31/13],    # Port 3
                         [30/14, 32/14, 44/13, 24/12]     # Port 3 (+100cm ommitted)
                 ]

averages = [44/30, 121/31, 68/31, 69/30, 46/29]   # averaged chi sq for each port depth
#'''

#''' # default
chisq_dof_AmBe = [[26/14, 267/14, 95/13, 280/14, 213/13],   # Port 5
                  [25/14, 66/14, 310/14, 22/14, 48/13],     # Port 1
                  [15/14, 39/14, 119/14, 34/14, 207/13],    # Port 3
                         [32/14, 27/14, 86/13, 38/12]       # Port 3 (+100cm ommitted)
                 ]

averages = [45/30, 310/31, 533/31, 296/30, 371/29]   # averaged chi sq for each port depth
#'''


plot_chi2_AmBe(chisq_dof_AmBe, labels, colors, averages)


print('done')

done
